# MAGAT — run everything from here

One cell per stage. Every cell is independently re-runnable: setup cells skip themselves once their
outputs exist, and every stage ends with a `bundle(...)` cell. Run output lives under `~/magat_runs`
(or `MAGAT_DATA`), **outside the clone**.

## Parameters and helpers

Run this first, always. It finds the repository on its own (a Jupyter kernel's working directory is
the notebook's folder, not where the server was launched) and puts the data root outside the clone.

In [ ]:
import os, sys, subprocess, pathlib, datetime, zipfile, glob

MAGAT_ROOT = ""     # leave empty: auto-detected. Set an absolute path only if the banner says so.

def _find_root():
    if MAGAT_ROOT:
        return pathlib.Path(MAGAT_ROOT).resolve()
    here = pathlib.Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / "magat.py").is_file() and (base / "core" / "src").is_dir():
            return base
        for child in base.glob("*"):
            if child.is_dir() and (child / "magat.py").is_file() and (child / "core" / "src").is_dir():
                return child
    raise RuntimeError(
        f"Cannot find the MAGAT repository at or above {here}. A Jupyter kernel's working directory "
        "is the notebook's own folder (and it keeps the OLD folder if the file was moved without a "
        "kernel restart). Restart the kernel, or set MAGAT_ROOT at the top of this cell.")

ROOT = _find_root()
os.chdir(ROOT)

# Everything a run produces lives OUTSIDE the clone, so a merge and a re-clone never touch it.
DATA = pathlib.Path(os.environ.get("MAGAT_DATA") or (pathlib.Path.home() / "magat_runs")).resolve()
os.environ["MAGAT_DATA"] = str(DATA)
LOGS = DATA / "logs"
BUNDLES = DATA / "bundles"
for _d in (LOGS, BUNDLES):
    _d.mkdir(parents=True, exist_ok=True)
assert ROOT not in DATA.parents and DATA != ROOT, f"data root {DATA} must be outside the clone {ROOT}"

GAME = "tribes"
PY = sys.executable
PUBLISHED_PORTFOLIO = ROOT / "games" / GAME / "portfolio" / "v1"
# Workers for parallel stages. Half the logical CPUs (SMT siblings add heat, not throughput) less two
# for the machine; MAGAT_WORKERS overrides. `magat machine` reports the measured ceiling.
WORKERS = int(os.environ.get("MAGAT_WORKERS") or max(1, (os.cpu_count() or 2) // 2 - 2))
# This machine's own currency: <data>/calibration/<game>-<name>. The published R7615 currency stays in
# the repository; a new calibration of the R7615 is published beside it under this name, never over it.
# A currency keeps the levels it was first measured at, so a currency at the published levels
# (games/<game>/levels.json) has a name of its own.
CURRENCY_NAME = os.environ.get("MAGAT_CURRENCY") or "r7615-time"

def magat(*args, check=True):
    """Run one launcher command; the launcher tees its own output to <DATA>/logs/<name>.log."""
    cmd = [PY, str(ROOT / "magat.py"), *map(str, args)]
    print("$ " + " ".join(cmd[2:]), flush=True)
    rc = subprocess.call(cmd)
    if check and rc != 0:
        raise RuntimeError(f"exit {rc}: magat {' '.join(map(str, args))}")
    return rc

def latest_log(name):
    txt = (LOGS / f"{name}.log").read_text(encoding="utf-8", errors="replace")
    return txt.rsplit("\n>>>>> ", 1)[-1]

def assert_in_log(name, needle):
    assert needle in latest_log(name), f"'{needle}' not found in the latest {name} run - read {LOGS / (name + '.log')}"
    print(f"[gate] '{needle}' confirmed in {name}")

def summary(results, until="## Budget multiplier"):
    """Print an analysis's SUMMARY.md up to a section (the provenance, the side by side and the headline)."""
    text = (pathlib.Path(results) / "SUMMARY.md").read_text(encoding="utf-8")
    cut = text.find(until)
    print(text if cut < 0 else text[:cut] + f"... the rest in {pathlib.Path(results) / 'SUMMARY.md'}")

def bundle(name, items, exclude=("seqlog_", "mutlog_", "_running.lock")):
    """Zip the pre-specified files of a stage into <DATA>/bundles/<name>.zip."""
    out = BUNDLES / f"{name}.zip"
    if out.exists():
        out.unlink()
    files = []
    for it in items:
        p = pathlib.Path(it)
        if not p.is_absolute():
            p = DATA / p
        if p.is_dir():
            files += [q for q in p.rglob("*") if q.is_file()]
        elif p.exists():
            files.append(p)
        else:
            files += [pathlib.Path(h) for h in glob.glob(str(p)) if pathlib.Path(h).is_file()]
    files = sorted({f for f in files if not any(x in f.name for x in exclude)})
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for f in files:
            z.write(f, str(f.relative_to(DATA)) if DATA in f.parents else f.name)
    print(f"{len(files)} files -> {out}")
    return out

print("repository :", ROOT)
print("data root  :", DATA, "(outside the clone)")
print("python     :", PY)
print("workers    :", WORKERS)

## Build, tests, layering

In [ ]:
magat("build", "--tests")
magat("test")
magat("layering")

## Portfolio

Build this machine's own copy of the portfolio from the game's scenario sources, freeze its split,
and verify it against the published one: the same identity, every manifest row equal, and the same
plans from both sets of scenario files. A rebuild that matches is the evidence that positions, and
therefore every published result, reproduce here. Skips itself once the portfolio is built and frozen;
the journal makes an interrupted build resume where it stopped.

In [ ]:
PORTFOLIO = DATA / "portfolio" / GAME
if (PORTFOLIO / "_split_dev.txt").exists() and (PORTFOLIO / "_split_holdout.txt").exists():
    print(f"[skip] {PORTFOLIO} is built and frozen")
else:
    magat("portfolio", "build", "--game", GAME, "--out", PORTFOLIO, "--threads", WORKERS)
    magat("portfolio", "freeze", "--portfolio", PORTFOLIO)
print((PORTFOLIO / "split_report.txt").read_text(encoding="utf-8"))
magat("portfolio", "verify", "--game", GAME, "--portfolio", PORTFOLIO, "--published", PUBLISHED_PORTFOLIO, "--plan", 4)
assert_in_log("portfolio-verify", "VERIFY: PASS")
bundle("portfolio", [PORTFOLIO / f for f in ("portfolio.csv", "_split_dev.txt", "_split_holdout.txt",
                                             "identity.txt", "split_report.txt")]
       + [LOGS / "portfolio-build.log", LOGS / "portfolio-freeze.log", LOGS / "portfolio-verify.log"])

## Machine, threads, calibration

The machine banner first: physical cores, the worker ceiling and why. Then the thread sweep, which
picks the knee of sustained throughput per budget level (results do not depend on it, only wall-clock
does). Each skips itself once its output exists.

In [ ]:
magat("machine")

### The levels

A budget level is a number of compute units, chosen so that it is a round amount of thinking time on
the reference machine: 500, 1000 and 5000 ms of a whole turn, single-threaded on the R7615, averaged
over positions spread across the portfolio. They are published with the repository
(`games/<game>/levels.json`, with the record behind them), and every currency, check and experiment
below runs at them. This cell reads them; it never measures them. Measuring them again is a
maintainer's job on the reference machine (`notebooks/maintainers/measure_levels.ipynb`), because new
levels mean every currency must be measured again.

In [ ]:
# The budget levels, read from the game's published games/<game>/levels.json: the units that are 500, 1000
# and 5000 ms of a whole turn (its initialization included), single-threaded on the reference machine,
# averaged over positions spread across the portfolio (docs/CURRENCY.md, "Units as time"). This notebook
# never measures them: every currency is measured at the published levels, whatever machine it is on.
import json
LEVELS_FILE = ROOT / "games" / GAME / "levels.json"
assert LEVELS_FILE.is_file(), f"{LEVELS_FILE} is missing: the levels are published with the repository (pull it again)"
LEVELS_DOC = json.loads(LEVELS_FILE.read_text(encoding="utf-8"))
LEVEL_UNITS = [int(lv["units"]) for lv in LEVELS_DOC["levels"]]
LEVELS = ",".join(map(str, LEVEL_UNITS))
MID_LEVEL = LEVEL_UNITS[len(LEVEL_UNITS) // 2]     # the level single-level checks run at
print(f"levels of {GAME}, measured {LEVELS_DOC['measured']} on {LEVELS_DOC['machine'].split(' | ')[0]}:")
for lv in LEVELS_DOC["levels"]:
    print(f"  {lv['units']:>9,} units = {lv['time_ms']:>5,} ms of thinking on the reference machine (measured {lv['measured']['mean_ms']:,.0f} ms)")

In [ ]:
# threads.json goes beside this machine's currency: that is where a run looks for it.
CALIBRATION = DATA / "calibration" / f"{GAME}-{CURRENCY_NAME}"
REDO_THREADS = False   # True: sweep again; the previous threads.json and thread_sweep.csv are kept, renamed
if REDO_THREADS:
    stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    for f in ("threads.json", "thread_sweep.csv"):
        if (CALIBRATION / f).exists():
            (CALIBRATION / f).rename(CALIBRATION / f"{f}.before-{stamp}")
            print(f"[redo] kept the previous {f} as {f}.before-{stamp}")
if (CALIBRATION / "threads.json").exists():
    print(f"[skip] {CALIBRATION / 'threads.json'} exists")
else:
    magat("threads", "--game", GAME, "--portfolio", PORTFOLIO, "--out", CALIBRATION, "--levels", LEVELS)
print((CALIBRATION / "threads.json").read_text(encoding="utf-8"))
bundle("threads", [CALIBRATION / "threads.json", CALIBRATION / "thread_sweep.csv", LOGS / "threads.log"])

### This machine's currency, in one session

`calibrate-controlled` prices every arm at the levels above in one calibration session, which writes all
or nothing. First it waits (up to `CAL_WAIT_QUIET` minutes) for the machine to be quiet and checks that
two estimates agree. Then it measures, and gives a verdict:
- **PASS**: the calibrator's gates passed and nothing was seen disturbing it.
- **SUSPECT**: the gates passed, but the machine was busy with other work (`CAL_OTHER_CPUS`) or the
  container was CPU-throttled. The currency is written; to measure it again when the machine is quiet,
  give `CURRENCY_NAME` a new name.
- **FAIL**: nothing was written. It is retried once, and running the cell again measures it again.

Then it checks the new currency against itself (did the machine hold its speed?), and against the
published currency once there is one. The measuring cell skips itself once the currency prices every
reference arm at every level. Over a currency that lacks an arm, measured before that arm was registered,
it prices just the missing arms and merges them in by ratio; the preflight cell says so first. Keep
this tab open while it runs, and start nothing else in this server. The screen shows the verdict;
`record/SUMMARY.md` in the currency's folder has the rest, and the last cell bundles it all.

In [ ]:
# This machine's own currency, on this machine's own portfolio, in one session (calibrate-controlled; read
# docs/CALIBRATING_A_MACHINE.md, "On a shared server"). Idle and single-threaded: nothing else of yours may
# run while it measures. The currency prices every operation an arm declares, its own work included.
CAL_OTHER_CPUS = 4.0    # other work on the machine, in logical CPUs, that makes the session SUSPECT
CAL_WAIT_QUIET = 30     # minutes to wait for the machine to drop below that before refusing to start
CAL_SOAK_MINUTES = 0    # a server needs no warming up: the stability check is two estimates that agree within 3%
WEIGHTS = CALIBRATION / "compute_weights.properties"     # the currency every run below spends
CAL_ARGS = ["--name", CURRENCY_NAME, "--game", GAME, "--portfolio", PORTFOLIO, "--other-cpus", CAL_OTHER_CPUS,
            "--wait-quiet", CAL_WAIT_QUIET, "--soak-minutes", CAL_SOAK_MINUTES, "--levels", LEVELS]

def priced():
    """Whether the currency prices every reference arm at every level: the check of the last calibration cell."""
    return WEIGHTS.is_file() and magat("weights", "--game", GAME, "--weights", WEIGHTS, "--arms", "@reference",
                                       "--levels", LEVELS, check=False) == 0

incomplete = WEIGHTS.is_file() and not priced()
magat("calibrate-controlled", *CAL_ARGS, "--preflight")
if incomplete:
    print(f"\n*** {WEIGHTS} exists but does not price every reference arm: the next cell prices the missing ones in "
          "one session and merges them into it by ratio. To measure a whole new currency in one session instead, "
          "give CURRENCY_NAME a new name in the first cell and run the cells from there again.")

In [ ]:
# The session, then the checks. Skips itself once the currency prices every reference arm at every level. Over
# a currency that lacks one, calibrate-controlled prices what is missing and merges it in by ratio. A session
# that did not finish wrote nothing, so running the cell again measures it again.
if priced():
    print(f"[skip] {WEIGHTS} prices every reference arm at every level")
else:
    rc = magat("calibrate-controlled", *CAL_ARGS, check=False)
    if rc == 5:
        print("\n*** SUSPECT: the currency is written, but read the reasons above. To measure it again when the machine "
              "is quiet, give CURRENCY_NAME a new name in the first cell and run it and the calibration cells again.")
    elif rc == 4:
        raise RuntimeError("the machine stayed busy and nothing was started: run this cell again later")
    elif rc != 0:
        raise RuntimeError(f"exit {rc}: run this cell again; the record is {CALIBRATION / 'record' / 'SUMMARY.md'}")
summary(CALIBRATION / "record", until="## ")

In [ ]:
# Every reference arm priced at every level, then the bundle: the currency and its whole record.
magat("weights", "--game", GAME, "--weights", WEIGHTS, "--arms", "@reference", "--levels", LEVELS)
bundle("calibration", [CALIBRATION, LOGS / "calibrate-controlled.log"])

## Gates and probes

The contract laws on published positions; the equivalence gate, where every ported arm must reproduce
the engine agent it was ported from, field for field; and the audit, where each arm's declared
operations and forward model must account for its turn. Nothing downstream runs otherwise.

In [ ]:
magat("check", "contract", "--game", GAME, "--scenarios", 8)
assert_in_log("check-contract", "CONTRACT: PASS")
magat("check", "equivalence", "--game", GAME, "--scenarios", 8, "--visits", 2, "--budgets", "8000,40000")
assert_in_log("check-equivalence", "EQUIVALENCE: PASS")
# The audit: each arm's declared operations and forward model account for its turn (fails above 25% unaccounted).
magat("check", "audit", "--game", GAME, "--portfolio", PORTFOLIO, "--arms", "@reference", "--budget", MID_LEVEL)
assert_in_log("check-audit", "AUDIT: PASS")
bundle("gates", [LOGS / f"{n}.log" for n in ("check-contract", "check-equivalence", "check-audit", "test")])

## Experiment: reference

The four reference algorithms and the random floor at the published levels (games/tribes/levels.json: 500, 1000 and 5000 ms of a whole turn on the R7615), on the whole portfolio: the field every new arm is compared against.

Split `all`; baseline `emcts`; levels (budget x visits) 20,000 x 8, 9,500 x 8, 101,500 x 6; arms random, mcts, emcts, oep, tbets
(110 turns per scenario of the split). `experiment run` validates the spec, runs it, analyses it,
prints the headline and bundles the run and its report. Re-running the cell resumes: finished leaves are
skipped, a partial leaf continues where it stopped.

In [ ]:
SPEC = ROOT / "experiments" / GAME / "reference.json"
RUN = DATA / GAME / "runs" / "reference"
RESULTS = DATA / GAME / "results" / "reference"
magat("experiment", "run", SPEC, "--split", "all", "--portfolio", PORTFOLIO, "--weights", WEIGHTS, "--out", RUN,
      "--results", RESULTS)

## Shakedown

Every moving part at toy size, minutes, exits non-zero on any failure. Run before every pull request.

In [ ]:
magat("shakedown")
bundle("shakedown", [LOGS / "shakedown.log"])

## Inventory

What to keep from this machine: every bundle the stages above wrote, with a tick, the files behind
them, and the calibration's verdict. Download the bundles; everything else can be regenerated from
them.

In [ ]:
# Defaults, so this cell also runs on its own after a kernel restart.
PORTFOLIO = globals().get("PORTFOLIO", DATA / "portfolio" / GAME)
CALIBRATION = globals().get("CALIBRATION", DATA / "calibration" / f"{GAME}-{CURRENCY_NAME}")
BUNDLED = [
    ("portfolio", "this machine's portfolio: manifest, split files, identity, the verify log"),
    ("threads", "the thread sweep and threads.json"),
    ("calibration", f"the currency {CURRENCY_NAME}: weights, the session's record and verdict, the checks"),
    ("gates", "contract, equivalence, audit"),
    ("run-reference", "experiment reference: its run tree and its analysis"),
    ("shakedown", "the shakedown's log"),
]
FILES = [
    (PORTFOLIO / "identity.txt", "the portfolio's identity"),
    (CALIBRATION / "compute_weights.properties", "this machine's currency"),
    (CALIBRATION / "record" / "SUMMARY.md", "the calibration's verdict"),
    (CALIBRATION / "threads.json", "the thread choices runs read"),
    (DATA / GAME / "results" / "reference" / "SUMMARY.md", "the report of reference"),
]
total, missing = 0, []
print(f"Bundles in {BUNDLES}: download these; together they hold everything the analysis needs.")
for name, what in BUNDLED:
    path = BUNDLES / f"{name}.zip"
    size = path.stat().st_size if path.is_file() else 0
    total += size
    if not size:
        missing.append(name)
    print(f"  [{'ok' if size else 'missing':7}] {path.name:32} {size / 2**20:8.1f} MB   {what}")
print(f"  {total / 2**20:.1f} MB in all")
print("\nThe files behind them:")
for path, what in FILES:
    print(f"  [{'ok' if path.exists() else 'missing':7}] {path}   ({what})")
record = CALIBRATION / "record" / "SUMMARY.md"
if record.is_file():
    verdict = next((line for line in record.read_text(encoding="utf-8").splitlines() if line.startswith("- Verdict:")), None)
    print("\nCalibration, latest invocation: " + (verdict[len("- Verdict: "):] if verdict else f"read {record}"))
print("\nINVENTORY: " + ("COMPLETE" if not missing else f"{len(missing)} bundle(s) missing: " + ", ".join(missing)))